In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
import torch
import gc

gc.collect()

act1 = torch.load(
    "/content/drive/MyDrive/layer6_activations.pt"
).float()
act1 = act1[:50000]
print("act1 loaded:", act1.shape)

act2 = torch.load(
    "/content/drive/MyDrive/activations_finetuned_wiki.pt"
).float()
act2 = act2[:50000]
print("act2 loaded:", act2.shape)

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim=768, dict_size=6144, k=30):
        super().__init__()
        self.k = k
        self.encoder = nn.Linear(input_dim, dict_size)
        self.decoder = nn.Linear(dict_size, input_dim, bias=False)
        self.pre_bias = nn.Parameter(torch.zeros(input_dim))

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.data.norm(dim=0, keepdim=True)
        self.decoder.weight.data = (
            self.decoder.weight.data / norms.clamp(min=1e-8)
        )

    def forward(self, x):
        x_centered = x - self.pre_bias
        pre_activations = self.encoder(x_centered)
        topk_vals, topk_idx = torch.topk(pre_activations, self.k, dim=-1)
        features = torch.zeros_like(pre_activations)
        features.scatter_(-1, topk_idx, torch.relu(topk_vals))
        reconstruction = self.decoder(features) + self.pre_bias
        return reconstruction, features

In [ ]:
sae1 = SparseAutoencoder().to(device)
sae1.load_state_dict(
    torch.load("/content/drive/MyDrive/sae_layer6_base.pt",
    map_location=device)
)
sae1.eval()

sae2 = SparseAutoencoder().to(device)
sae2.load_state_dict(
    torch.load("/content/drive/MyDrive/sae_layer6_finetuned_v2.pt",
    map_location=device)
)
sae2.eval()
print("Both SAEs loaded.")

In [ ]:

dec1 = sae1.decoder.weight.T
dec2 = sae2.decoder.weight.T

# cosine similarity between each feature pair
cos_sim = torch.nn.functional.cosine_similarity(
    dec1, dec2, dim=-1
)

print("Mean cosine similarity:", cos_sim.mean().item())
print("Std:", cos_sim.std().item())
print("Fraction unchanged (sim > 0.9):", (cos_sim > 0.9).float().mean().item())
print("Fraction changed (sim < 0.5):", (cos_sim < 0.5).float().mean().item())

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(cos_sim.detach().cpu().numpy(), bins=100, edgecolor='black')
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Features")
plt.title("Feature Similarity: Base vs Finetuned SAE")
plt.axvline(x=0.9, color='green', linestyle='--', label='Unchanged (>0.9)')
plt.axvline(x=0.5, color='red', linestyle='--', label='Changed (<0.5)')
plt.legend()
plt.savefig("/content/drive/MyDrive/cosine_similarity_dist.png", dpi=150)
plt.show()
print("Plot saved.")

In [ ]:
from scipy.optimize import linear_sum_assignment

n = 512

dec1_sample = dec1[:n].detach().cpu().numpy()
dec2_sample = dec2[:n].detach().cpu().numpy()

dec1_norm = dec1_sample / np.linalg.norm(dec1_sample, axis=1, keepdims=True)
dec2_norm = dec2_sample / np.linalg.norm(dec2_sample, axis=1, keepdims=True)

sim_matrix = dec1_norm @ dec2_norm.T

row_ind, col_ind = linear_sum_assignment(-sim_matrix)
matched_sims = sim_matrix[row_ind, col_ind]

print("After matching:")
print("Mean similarity:", matched_sims.mean())
print("Fraction unchanged (>0.9):", (matched_sims > 0.9).mean())
print("Fraction changed (<0.5):", (matched_sims < 0.5).mean())

plt.figure(figsize=(10, 5))
plt.hist(matched_sims, bins=50, edgecolor='black')
plt.xlabel("Cosine Similarity (after matching)")
plt.ylabel("Number of Features")
plt.title("Matched Feature Similarity: Base vs Finetuned SAE")
plt.axvline(x=0.9, color='green', linestyle='--', label='Unchanged (>0.9)')
plt.axvline(x=0.5, color='red', linestyle='--', label='Changed (<0.5)')
plt.legend()
plt.savefig("/content/drive/MyDrive/matched_similarity_dist.png", dpi=150)
plt.show()

In [ ]:

batch_size = 1024
all_feats1 = []
all_feats2 = []

with torch.no_grad():
    for i in range(0, 50000, batch_size):
        x1 = act1[i:i+batch_size]
        x2 = act2[i:i+batch_size]
        _, f1 = sae1(x1)
        _, f2 = sae2(x2)
        all_feats1.append(f1.cpu())
        all_feats2.append(f2.cpu())

feats1 = torch.cat(all_feats1, dim=0)
feats2 = torch.cat(all_feats2, dim=0)

print("Features shape:", feats1.shape, feats2.shape)

freq1 = (feats1 > 0).float().mean(dim=0)
freq2 = (feats2 > 0).float().mean(dim=0)

mag1 = feats1.mean(dim=0)
mag2 = feats2.mean(dim=0)

print("Done computing feature stats.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# frequency comparison
axes[0].scatter(freq1.numpy(), freq2.numpy(), alpha=0.1, s=5)
axes[0].plot([0, freq1.max()], [0, freq1.max()], 'r--', label='no change')
axes[0].set_xlabel("Base SAE feature frequency")
axes[0].set_ylabel("Finetuned SAE feature frequency")
axes[0].set_title("Feature Activation Frequency Shift")
axes[0].legend()

# magnitude comparison
axes[1].scatter(mag1.numpy(), mag2.numpy(), alpha=0.1, s=5)
axes[1].plot([0, mag1.max()], [0, mag1.max()], 'r--', label='no change')
axes[1].set_xlabel("Base SAE feature magnitude")
axes[1].set_ylabel("Finetuned SAE feature magnitude")
axes[1].set_title("Feature Activation Magnitude Shift")
axes[1].legend()

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/feature_frequency_shift.png", dpi=150)
plt.show()

# quantify shifts
freq_diff = (freq2 - freq1).abs()
print("Top 10 features with biggest frequency increase:")
print(torch.topk(freq2 - freq1, 10).indices.tolist())
print("Top 10 features with biggest frequency decrease:")
print(torch.topk(freq1 - freq2, 10).indices.tolist())

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")

def get_top_tokens(feature_idx, features_matrix, act_matrix, tokenizer, top_n=10):

    feature_acts = features_matrix[:, feature_idx]
    top_indices = torch.topk(feature_acts, top_n).indices

    print(f"\nFeature {feature_idx}:")
    print(f"  Mean activation: {feature_acts.mean():.4f}")
    print(f"  Activation frequency: {(feature_acts > 0).float().mean():.4f}")

# most increased features (gained after finetuning)
increased = [4100, 3679, 777, 131, 4648, 5384, 2673, 1310, 313, 2722]
# most decreased features (lost after finetuning)
decreased = [5494, 991, 5993, 726, 3606, 3254, 5864, 490, 3302, 3218]

print(" FEATURES THAT INCREASED (new after finetuning) ")
for idx in increased[:5]:
    f1_val = freq1[idx].item()
    f2_val = freq2[idx].item()
    print(f"Feature {idx}: base={f1_val:.4f} → finetuned={f2_val:.4f} (Δ={f2_val-f1_val:+.4f})")

print("\nFEATURES THAT DECREASED (suppressed after finetuning) ")
for idx in decreased[:5]:
    f1_val = freq1[idx].item()
    f2_val = freq2[idx].item()
    print(f"Feature {idx}: base={f1_val:.4f} → finetuned={f2_val:.4f} (Δ={f2_val-f1_val:+.4f})")

fig, axes = plt.subplots(2, 5, figsize=(16, 6))

for i, idx in enumerate(increased[:5]):
    axes[0, i].bar(['Base', 'Finetuned'], [freq1[idx].item(), freq2[idx].item()],
                    color=['blue', 'orange'])
    axes[0, i].set_title(f'Feature {idx}\n(increased)')
    axes[0, i].set_ylabel('Frequency')

for i, idx in enumerate(decreased[:5]):
    axes[1, i].bar(['Base', 'Finetuned'], [freq1[idx].item(), freq2[idx].item()],
                    color=['blue', 'orange'])
    axes[1, i].set_title(f'Feature {idx}\n(decreased)')
    axes[1, i].set_ylabel('Frequency')

plt.suptitle('Top Changed Features: Base vs Finetuned')
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/top_changed_features.png", dpi=150)
plt.show()

In [ ]:

from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")

wiki = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)


all_tokens = []
for i in range(5000):
    text = wiki[i]["text"]
    if len(text.strip()) == 0:
        continue
    ids = tokenizer(
        text,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )["input_ids"][0]
    all_tokens.extend(ids.tolist())

all_tokens = all_tokens[:50000]
print(f"Total tokens: {len(all_tokens)}")

def show_top_tokens(feature_idx, features_matrix, tokens, tokenizer, top_n=10):
    acts = features_matrix[:, feature_idx]
    top_idx = torch.topk(acts, min(top_n, len(acts))).indices
    top_tokens = [tokenizer.decode([tokens[i]]) for i in top_idx.tolist()]
    top_vals = acts[top_idx].tolist()
    print(f"\nFeature {feature_idx} — top activating tokens:")
    for tok, val in zip(top_tokens, top_vals):
        print(f"  '{tok}' : {val:.3f}")

print(" INCREASED FEATURES (new after finetuning) ")
for idx in [4100, 3679, 777]:
    show_top_tokens(idx, feats2, all_tokens, tokenizer)

print("\n DECREASED FEATURES (suppressed after finetuning) ")
for idx in [5494, 991, 5993]:
    show_top_tokens(idx, feats1, all_tokens, tokenizer)

In [ ]:

print("=" * 50)
print("FEATURE COMPARISON SUMMARY")
print("=" * 50)

total_features = 6144
freq_diff = freq2 - freq1

# features that newly appeared (base ~0, finetuned > 0.1)
new_features = ((freq1 < 0.01) & (freq2 > 0.1)).sum().item()

# features that were suppressed (base > 0.1, finetuned ~0)
suppressed_features = ((freq1 > 0.1) & (freq2 < 0.01)).sum().item()

# features that stayed roughly same
stable_features = ((freq1 - freq2).abs() < 0.05).sum().item()

# features that increased moderately
increased_features = ((freq2 - freq1) > 0.05).sum().item()

# features that decreased moderately
decreased_features = ((freq1 - freq2) > 0.05).sum().item()

print(f"Total features: {total_features}")
print(f"Newly appeared (base≈0 → ft>0.1): {new_features} ({100*new_features/total_features:.1f}%)")
print(f"Suppressed (base>0.1 → ft≈0): {suppressed_features} ({100*suppressed_features/total_features:.1f}%)")
print(f"Stable (|Δ| < 0.05): {stable_features} ({100*stable_features/total_features:.1f}%)")
print(f"Moderately increased: {increased_features} ({100*increased_features/total_features:.1f}%)")
print(f"Moderately decreased: {decreased_features} ({100*decreased_features/total_features:.1f}%)")


print(f"\nBase SAE mean activation frequency: {freq1.mean():.4f}")
print(f"Finetuned SAE mean activation frequency: {freq2.mean():.4f}")
print(f"Overall frequency change: {(freq2.mean()-freq1.mean()):+.4f}")


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(
    ['New', 'Suppressed', 'Stable', 'Increased', 'Decreased'],
    [new_features, suppressed_features, stable_features,
     increased_features, decreased_features],
    color=['green', 'red', 'gray', 'orange', 'blue']
)
axes[0].set_title('Feature Change Categories')
axes[0].set_ylabel('Number of Features')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(freq1.numpy(), bins=50, alpha=0.6, label='Base', color='blue')
axes[1].hist(freq2.numpy(), bins=50, alpha=0.6, label='Finetuned', color='orange')
axes[1].set_title('Feature Frequency Distribution')
axes[1].set_xlabel('Activation Frequency')
axes[1].set_ylabel('Count')
axes[1].legend()

axes[2].hist((freq2 - freq1).numpy(), bins=50, color='purple', edgecolor='black')
axes[2].axvline(x=0, color='red', linestyle='--')
axes[2].set_title('Frequency Change (Finetuned - Base)')
axes[2].set_xlabel('Δ Frequency')
axes[2].set_ylabel('Count')

plt.suptitle('SAE Feature Comparison: Base vs Medical Finetuned Pythia-160M')
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/feature_comparison_summary.png", dpi=150)
plt.show()